# Day D1 — BOLD 신호와 fMRI 데이터의 생김새: 4D 배열, voxel, NIfTI, BIDS

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/D-fmri/day-D1.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/D-fmri/day-D1.md)

In [ ]:
# Colab 기본 런타임에 없는 패키지 설치
%pip install -q nibabel

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 4D 배열, voxel 시계열, 시간 × voxel 행렬

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
nx, ny, nz, nt = 8, 8, 4, 120              # 아주 작은 가상 뇌: 8×8×4 voxel, 120 시점
data = 1000 + rng.normal(0, 10, size=(nx, ny, nz, nt))   # 4D 배열 (x, y, z, 시간)

print("4D shape:", data.shape)
print("한 시점의 3D 영상:", data[..., 0].shape)
ts = data[3, 4, 2, :]                       # 한 voxel의 시계열
print("voxel (3,4,2) 시계열 길이:", ts.shape, "평균:", round(ts.mean(), 1))

# 분석 프로그램은 보통 (시간 × voxel) 2D 행렬로 바꿔 계산한다
mask = np.zeros((nx, ny, nz), dtype=bool)
mask[2:6, 2:6, :] = True                    # 가상 '뇌 마스크'
Y = data[mask].T                            # (voxel, 시간) → 전치 → (시간, voxel)
print("마스크 안 voxel 수:", mask.sum(), "| 2D 행렬:", Y.shape)

# 저장 형식에 따른 크기: 실제 크기의 4D 영상을 가정해 계산만 한다
shape_real = (64, 64, 36, 200)
n = np.prod(shape_real)
for dtype in ["int16", "float32", "float64"]:
    mb = n * np.dtype(dtype).itemsize / 1024**2
    print(f"{dtype:>8}: {mb:7.1f} MB")

### 예시 2 — affine으로 voxel 번호를 mm 좌표로 바꾸기

In [ ]:
import numpy as np

# affine: voxel 번호(i, j, k) → 스캐너/표준 공간 좌표(mm)를 잇는 4×4 행렬
voxel_size = 3.0
affine = np.array([
    [-voxel_size, 0, 0,  96.0],   # x: i가 커질수록 x(mm)가 작아진다 = 왼쪽→오른쪽이 뒤집힌 저장
    [0, voxel_size, 0, -132.0],
    [0, 0, voxel_size,  -78.0],
    [0, 0, 0, 1.0],
])

def vox2mm(ijk):
    return (affine @ np.append(ijk, 1))[:3]

def mm2vox(xyz):
    return (np.linalg.inv(affine) @ np.append(xyz, 1))[:3]

for ijk in [(0, 0, 0), (32, 44, 26), (63, 0, 0)]:
    print(ijk, "→", vox2mm(np.array(ijk)), "mm")

xyz = vox2mm(np.array([32, 44, 26]))
back = mm2vox(xyz)
print("되돌리기:", back)
assert np.allclose(back, [32, 44, 26])      # 왕복 변환이 맞는지 자기 점검

# 같은 배열을 '왼쪽이 +x'로 잘못 해석하면?
wrong = affine.copy(); wrong[0, 0] = +voxel_size; wrong[0, 3] = -96.0
print("올바른 x:", vox2mm(np.array([10, 44, 26]))[0],
      "| 부호를 잘못 읽은 x:", (wrong @ np.array([10, 44, 26, 1]))[0])

### 예시 3 — BIDS 이름 읽기와 자료형에 따른 정보 손실

In [ ]:
import json
import numpy as np

# (가) BIDS 파일 이름에서 정보 꺼내기: key-value 조각이 '_'로 이어진다
names = [
    "sub-01/func/sub-01_task-rest_run-1_bold.nii.gz",
    "sub-01/func/sub-01_task-nback_run-2_bold.nii.gz",
    "sub-02/anat/sub-02_T1w.nii.gz",
]
for path in names:
    stem = path.split("/")[-1].split(".")[0]
    *entities, suffix = stem.split("_")
    info = dict(e.split("-", 1) for e in entities)
    print(f"{suffix:5} {info}")

# 영상 옆의 JSON 사이드카에는 분석에 꼭 필요한 획득 정보가 들어 있다 (가상 예시)
sidecar = json.loads('{"RepetitionTime": 2.0, "TaskName": "nback", '
                     '"SliceTiming": [0.0, 1.0, 0.2, 1.2, 0.4, 1.4, 0.6, 1.6, 0.8, 1.8]}')
print("TR =", sidecar["RepetitionTime"], "s | 슬라이스 수 =", len(sidecar["SliceTiming"]))

# (나) 저장 자료형과 정보 손실: 정수로 저장하면 소수점이 잘린다
rng = np.random.default_rng(0)
raw = rng.integers(200, 4000, size=10_000).astype(np.int16)   # 가상 원시 신호
scaled = raw * 0.037                                            # 어떤 처리 뒤의 값
as_float = scaled.astype(np.float32)
as_int = np.round(scaled).astype(np.int16)
print("서로 다른 값의 수 — 원본:", np.unique(raw).size,
      "| float32 저장:", np.unique(as_float).size,
      "| int16 저장:", np.unique(as_int).size)

### 참고 — 실제 NIfTI 파일을 읽을 때의 명령 형태 (nibabel 필요)

> 이 코드는 `pip install nibabel nilearn`으로 설치한 환경(nibabel 5.5, nilearn 0.14.1)에서, 같은 이름의 **가상 파일**을 만들어 그대로 실행해 오류 없이 도는 것을 확인했습니다. 출력은 여러분의 실제 파일에 따라 달라지므로 싣지 않았습니다.

In [ ]:
# nibabel이 필요하다: pip install nibabel
import nibabel as nib
img = nib.load("sub-01/func/sub-01_task-rest_run-1_bold.nii.gz")
print(img.shape)        # (x, y, z, 시간)
print(img.affine)       # voxel → mm 4×4 행렬
data = img.get_fdata()  # numpy 배열 (실수형)
print(img.header.get_zooms())   # voxel 크기(mm)와 TR(초)